# Descriptors
Descriptors are the engine behind many of Python's most powerful features, including `@property`, `classmethod`, `staticmethod`, and even regular methods. A descriptor is simply an object that implements at least one of the following methods in the **Descriptor Protocol**: `__get__`, `__set__`, or `__delete__`.

## A Simple Descriptor
Descriptors are defined as class-level attributes in another class.

In [1]:
class Ten:
    def __get__(self, obj, objtype=None):
        return 10

class MyClass:
    x = 5
    y = Ten() # Descriptor instance

obj = MyClass()
print(obj.x) # Returns 5
print(obj.y) # Returns 10 (via __get__)

5
10


## Data vs Non-Data Descriptors
- **Non-data descriptor**: Implements only `__get__`. It can be overridden by an instance attribute.
- **Data descriptor**: Implements both `__get__` and `__set__`. It always takes precedence over an instance attribute.

## Practical Example: Validation
Descriptors are excellent for reusable attribute validation logic.

In [2]:
class IntField:
    def __init__(self, name):
        self.name = f"_{name}"

    def __get__(self, obj, objtype):
        return getattr(obj, self.name)

    def __set__(self, obj, value):
        if not isinstance(value, int):
            raise TypeError(f"{self.name[1:]} must be an integer")
        if value < 0:
            raise ValueError(f"{self.name[1:]} must be positive")
        setattr(obj, self.name, value)

class User:
    age = IntField("age")

u = User()
u.age = 25
print(f"User age: {u.age}")

try:
    u.age = -5
except ValueError as e:
    print(e)

User age: 25
age must be positive


## `__set_name__` (Python 3.6+)
One historical pain point was that descriptors didn't know the name they were assigned to in the owner class. `__set_name__` fixes this.

In [3]:
class ValidString:
    def __set_name__(self, owner, name):
        self.private_name = f"_{name}"

    def __get__(self, obj, objtype):
        return getattr(obj, self.private_name)

    def __set__(self, obj, value):
        if not isinstance(value, str):
            raise TypeError("Must be a string")
        setattr(obj, self.private_name, value)

class Profile:
    username = ValidString() # No need to pass 'username' anymore

p = Profile()
p.username = "python_engineer"
print(p.username)

python_engineer


## Summary
- **Descriptor Protocol**: `__get__`, `__set__`, `__delete__`.
- **Scope**: They must be defined as class attributes.
- **Usage**: Ideal for validation, lazy-loading, and implementing custom methods or properties.
- **Note**: If you find yourself writing the same `@property` logic in multiple classes, use a Descriptor instead!